Exercice 1 — LoRALayer

In [80]:
import torch
import torch.nn as nn

class LoRALayer(nn.Module):
    def __init__(self, in_dim, out_dim, rank, alpha):
        super().__init__()
        std_dev    = 1 / torch.sqrt(torch.tensor(rank).float())
        self.A     = nn.Parameter(torch.randn(in_dim, rank) * std_dev)
        self.B     = nn.Parameter(torch.zeros(rank, out_dim))
        self.alpha = alpha

    def forward(self, x):
        # x @ A → (batch, rank)
        # (x @ A) @ B → (batch, out_dim)
        x = self.alpha * (x @ self.A @ self.B)
        return x


# Test
torch.manual_seed(123)
lora = LoRALayer(in_dim=4, out_dim=8, rank=2, alpha=1)
x    = torch.randn(1, 4)

print(f"Input shape  : {x.shape}")
print(f"Output shape : {lora(x).shape}")
print(f"Matrice A shape : {lora.A.shape}")
print(f"Matrice B shape : {lora.B.shape}")
print(f"Params LoRA : {lora.A.numel() + lora.B.numel()}")

Input shape  : torch.Size([1, 4])
Output shape : torch.Size([1, 8])
Matrice A shape : torch.Size([4, 2])
Matrice B shape : torch.Size([2, 8])
Params LoRA : 24


Exercice 2 — LinearWithLoRA

In [81]:
class LinearWithLoRA(nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.linear = linear
        self.lora   = LoRALayer(
            linear.in_features,
            linear.out_features,
            rank,
            alpha
        )

    def forward(self, x):
        # Sortie = couche linéaire originale + adaptation LoRA
        return self.linear(x) + self.lora(x)


# Test
torch.manual_seed(123)
layer        = nn.Linear(4, 8)
x            = torch.randn(1, 4)

print("Original output :", layer(x))
print("Layer :", layer)

# Appliquer LoRA
layer_lora_1 = LinearWithLoRA(layer, rank=2, alpha=4)
print("\nLoRA output :", layer_lora_1(x))
print("Layer avec LoRA :", layer_lora_1)

Original output : tensor([[ 0.1758,  1.2343,  0.6518, -0.3266, -0.0436, -0.5039, -0.3522, -1.0122]],
       grad_fn=<AddmmBackward0>)
Layer : Linear(in_features=4, out_features=8, bias=True)

LoRA output : tensor([[ 0.1758,  1.2343,  0.6518, -0.3266, -0.0436, -0.5039, -0.3522, -1.0122]],
       grad_fn=<AddBackward0>)
Layer avec LoRA : LinearWithLoRA(
  (linear): Linear(in_features=4, out_features=8, bias=True)
  (lora): LoRALayer()
)


Exercice 3 — Réseau simple avec LoRA

In [82]:
torch.manual_seed(123)

# Réseau monocouche simple
simple_layer = nn.Linear(4, 8)
x            = torch.randn(1, 4)

# Sortie originale
output_original = simple_layer(x)
print(f"Sortie originale : {output_original}")

# Remplacer par LinearWithLoRA
simple_layer_lora = LinearWithLoRA(simple_layer, rank=2, alpha=0)
# Note : avec alpha=0, LoRA n'ajoute rien → sortie identique

output_lora = simple_layer_lora(x)
print(f"Sortie avec LoRA (alpha=0) : {output_lora}")
print(f"Sorties identiques : {torch.allclose(output_original, output_lora)}")

Sortie originale : tensor([[ 0.1758,  1.2343,  0.6518, -0.3266, -0.0436, -0.5039, -0.3522, -1.0122]],
       grad_fn=<AddmmBackward0>)
Sortie avec LoRA (alpha=0) : tensor([[ 0.1758,  1.2343,  0.6518, -0.3266, -0.0436, -0.5039, -0.3522, -1.0122]],
       grad_fn=<AddBackward0>)
Sorties identiques : True


Exercice 4 — LinearWithLoRAMerged

In [83]:
import torch.nn.functional as F

class LinearWithLoRAMerged(nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.linear = linear
        self.lora   = LoRALayer(
            linear.in_features,
            linear.out_features,
            rank,
            alpha
        )

    def forward(self, x):
        # Fusionner A et B en une seule matrice
        lora_weight = self.lora.A @ self.lora.B

        # Combiner avec les poids originaux
        combined_weight = self.linear.weight + self.lora.alpha * lora_weight.T

        return F.linear(x, combined_weight, self.linear.bias)


# Test d'équivalence
torch.manual_seed(123)
layer        = nn.Linear(4, 8)
x            = torch.randn(1, 4)

layer_lora_1 = LinearWithLoRA(layer, rank=2, alpha=4)
layer_lora_2 = LinearWithLoRAMerged(layer, rank=2, alpha=4)

# Les deux doivent donner le même résultat
out1 = layer_lora_1(x)
out2 = layer_lora_2(x)

print(f"LinearWithLoRA       : {out1}")
print(f"LinearWithLoRAMerged : {out2}")
print(f"Résultats identiques : {torch.allclose(out1, out2, atol=1e-6)}")

LinearWithLoRA       : tensor([[ 0.1758,  1.2343,  0.6518, -0.3266, -0.0436, -0.5039, -0.3522, -1.0122]],
       grad_fn=<AddBackward0>)
LinearWithLoRAMerged : tensor([[ 0.1758,  1.2343,  0.6518, -0.3266, -0.0436, -0.5039, -0.3522, -1.0122]],
       grad_fn=<AddmmBackward0>)
Résultats identiques : True


Exercice 5 — MLP avec LoRA

In [84]:
class MultilayerPerceptron(nn.Module):
    def __init__(self, num_features, num_hidden_1, num_hidden_2, num_classes):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(num_features, num_hidden_1),  # couche 0
            nn.ReLU(),                               # couche 1
            nn.Linear(num_hidden_1, num_hidden_2),  # couche 2
            nn.ReLU(),                               # couche 3
            nn.Linear(num_hidden_2, num_classes)     # couche 4
        )

    def forward(self, x):
        x = self.layers(x)
        return x


# Hyperparamètres — MNIST : images 28×28 = 784 pixels, 10 chiffres
num_features = 784
num_hidden_1 = 128
num_hidden_2 = 256
num_classes  = 10
learning_rate = 0.0005
num_epochs    = 2

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = MultilayerPerceptron(
    num_features=num_features,
    num_hidden_1=num_hidden_1,
    num_hidden_2=num_hidden_2,
    num_classes=num_classes
)
model.to(DEVICE)

optimizer_pretrained = torch.optim.Adam(model.parameters(), lr=learning_rate)
print(f"Device : {DEVICE}")
print(model)

Device : cuda
MultilayerPerceptron(
  (layers): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=256, bias=True)
    (3): ReLU()
    (4): Linear(in_features=256, out_features=10, bias=True)
  )
)


Chargement MNIST

In [85]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

BATCH_SIZE = 64

train_dataset = datasets.MNIST(
    root='data', train=True,
    transform=transforms.ToTensor(), download=True
)
test_dataset = datasets.MNIST(
    root='data', train=False,
    transform=transforms.ToTensor(), download=True
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

for images, labels in train_loader:
    print(f"Image batch dimensions : {images.shape}")
    print(f"Label batch dimensions : {labels.shape}")
    break

100%|██████████| 9.91M/9.91M [00:00<00:00, 16.8MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 495kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.50MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 7.57MB/s]

Image batch dimensions : torch.Size([64, 1, 28, 28])
Label batch dimensions : torch.Size([64])


Fonctions d'évaluation et d'entraînement

In [86]:
import time

def compute_accuracy(model, data_loader, device):
    model.eval()
    correct_pred, num_examples = 0, 0
    with torch.no_grad():
        for features, targets in data_loader:
            features = features.view(-1, 784).to(device)
            targets  = targets.to(device)
            logits   = model(features)
            _, predicted_labels = torch.max(logits, 1)
            num_examples  += targets.size(0)
            correct_pred  += (predicted_labels == targets).sum().item()
    return correct_pred / num_examples * 100


def train(num_epochs, model, optimizer, train_loader, device):
    start_time = time.time()
    for epoch in range(num_epochs):
        model.train()
        for batch_idx, (features, targets) in enumerate(train_loader):
            features = features.view(-1, 784).to(device)
            targets  = targets.to(device)

            logits = model(features)
            loss   = F.cross_entropy(logits, targets)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            if not batch_idx % 400:
                print(f"Epoch: {epoch+1:03d}/{num_epochs} | "
                      f"Batch {batch_idx:03d}/{len(train_loader)} | "
                      f"Loss: {loss:.4f}")

        acc = compute_accuracy(model, train_loader, device)
        print(f"Epoch: {epoch+1:03d}/{num_epochs} training accuracy: {acc:.2f}%")
        print(f"Time elapsed: {(time.time()-start_time)/60:.2f} min")

    print(f"Total Training Time: {(time.time()-start_time)/60:.2f} min")


# Entraîner le modèle original
train(num_epochs, model, optimizer_pretrained, train_loader, DEVICE)
print(f"Test accuracy: {compute_accuracy(model, test_loader, DEVICE):.2f}%")

Epoch: 001/2 | Batch 000/938 | Loss: 2.3109
Epoch: 001/2 | Batch 400/938 | Loss: 0.4703
Epoch: 001/2 | Batch 800/938 | Loss: 0.1992
Epoch: 001/2 training accuracy: 95.03%
Time elapsed: 0.25 min
Epoch: 002/2 | Batch 000/938 | Loss: 0.1563
Epoch: 002/2 | Batch 400/938 | Loss: 0.1295
Epoch: 002/2 | Batch 800/938 | Loss: 0.1004
Epoch: 002/2 training accuracy: 96.64%
Time elapsed: 0.48 min
Total Training Time: 0.48 min
Test accuracy: 96.28%


Exercice 6 — Remplacer + Geler + Fine-tuner avec LoRA

In [87]:
import copy

# Copie du modèle original
model_lora = copy.deepcopy(model)

# Remplacer les couches Linear par LinearWithLoRAMerged
model_lora.layers[0] = LinearWithLoRAMerged(model_lora.layers[0], rank=4, alpha=8)
model_lora.layers[2] = LinearWithLoRAMerged(model_lora.layers[2], rank=4, alpha=8)
model_lora.layers[4] = LinearWithLoRAMerged(model_lora.layers[4], rank=4, alpha=8)
model_lora.to(DEVICE)

print(model_lora)
print(f"\nTest accuracy modèle original : {compute_accuracy(model, test_loader, DEVICE):.2f}%")
print(f"Test accuracy modèle LoRA     : {compute_accuracy(model_lora, test_loader, DEVICE):.2f}%")


# Geler les couches Linear originales
def freeze_linear_layers(model):
    for child in model.children():
        if isinstance(child, nn.Linear):
            for param in child.parameters():
                param.requires_grad = False
        else:
            freeze_linear_layers(child)

freeze_linear_layers(model_lora)

# Vérifier ce qui est entraînable
print("\n=== Paramètres entraînables ===")
total_params = 0
lora_params  = 0
for name, param in model_lora.named_parameters():
    trainable = param.requires_grad
    print(f"{name:40} → {'ENTRAÎNABLE' if trainable else 'GELÉ'}")
    total_params += param.numel()
    if trainable:
        lora_params += param.numel()

print(f"\nTotal params    : {total_params:,}")
print(f"Params LoRA     : {lora_params:,}")
print(f"Réduction       : {(1 - lora_params/total_params)*100:.1f}% de params en moins !")


# Fine-tuner seulement les couches LoRA
optimizer_lora = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model_lora.parameters()),
    lr=learning_rate
)

train(num_epochs, model_lora, optimizer_lora, train_loader, DEVICE)

print(f"\n=== Résultats finaux ===")
print(f"Test accuracy modèle original  : {compute_accuracy(model, test_loader, DEVICE):.2f}%")
print(f"Test accuracy LoRA fine-tuned  : {compute_accuracy(model_lora, test_loader, DEVICE):.2f}%")

MultilayerPerceptron(
  (layers): Sequential(
    (0): LinearWithLoRAMerged(
      (linear): Linear(in_features=784, out_features=128, bias=True)
      (lora): LoRALayer()
    )
    (1): ReLU()
    (2): LinearWithLoRAMerged(
      (linear): Linear(in_features=128, out_features=256, bias=True)
      (lora): LoRALayer()
    )
    (3): ReLU()
    (4): LinearWithLoRAMerged(
      (linear): Linear(in_features=256, out_features=10, bias=True)
      (lora): LoRALayer()
    )
  )
)

Test accuracy modèle original : 96.28%
Test accuracy modèle LoRA     : 96.28%

=== Paramètres entraînables ===
layers.0.linear.weight                   → GELÉ
layers.0.linear.bias                     → GELÉ
layers.0.lora.A                          → ENTRAÎNABLE
layers.0.lora.B                          → ENTRAÎNABLE
layers.2.linear.weight                   → GELÉ
layers.2.linear.bias                     → GELÉ
layers.2.lora.A                          → ENTRAÎNABLE
layers.2.lora.B                          → ENTRAÎNAB